# Build ROQ Basis for mlgw\_bns (GW170817)

This notebook builds **Reduced Order Quadrature (ROQ)** interpolants for the
**mlgw\_bns** waveform model using [JenpyROQ](https://github.com/GCArullo/JenpyROQ),
as described in the [mlgw\_bns paper](https://arxiv.org/abs/2210.15684).

## Why ROQ?

For GW170817 parameter estimation with a 128 s segment at 4 kHz sampling rate,
the full frequency grid has ~253,000 points. This makes likelihood evaluations
extremely expensive, limiting GPU-based samplers (e.g., SHARPy) to only ~80
particles on an NVIDIA A100. The ROQ compresses this grid down to O(1000)
empirical nodes, giving ~100–250× speedup in likelihood evaluation and allowing
O(500+) particles for robust sampling.

## Why NumPy (not JAX)?

JAX's LLVM JIT compilation requires significant memory overhead during the
ROQ basis construction phase (which generates tens of thousands of waveforms).
On machines with limited RAM (Colab: 12 GB, IGWN Jupyter: varies), this leads
to OOM errors. Using the original NumPy-based  model avoids JAX
entirely during this build phase, while the resulting ROQ basis can still be
used with JAX for parameter estimation.

**Target:** Google Colab / IGWN JupyterHub / any Linux machine with ≥8 GB RAM
**Runtime:** 2–6 hours depending on hardware
**Output:**  (linear + quadratic interpolants)

## 1. Environment Setup

This cell detects the runtime environment and installs the required packages.

### IGWN / LIGO JupyterHub users

**Recommended:** run the setup script once from a terminal before using this notebook:

```bash
cd ~/mlgw_bns_jax
bash setup_igwn.sh
```

Then select the **`mlgw-bns-jax`** kernel in this notebook and skip to cell 3.

If you prefer to set up within the notebook, the cell below will handle it.

### Common issues on IGWN (solved by `setup_igwn.sh`):
- `conda env create -f environment.yml` fails due to base env conflicts → uses `conda create` with `--override-channels`
- JAX gets upgraded to 0.9.x by NetKet / SHARPy → re-pinned to 0.4.38
- `np.VisibleDeprecationWarning` missing (numpy 2.0) → patched in JenpyROQ
- `pkg_resources` missing (setuptools ≥ 72) → pinned setuptools < 71

### Required packages (no JAX needed for the ROQ build itself):
- `mlgw_bns` — the original NumPy-based waveform model (installed from this repo)
- `JenpyROQ` — the ROQ basis builder
- `h5py`, `scikit-learn`, `matplotlib` — dependencies

In [ ]:
import os, subprocess, sys, shutil
import numpy as np

# ── Detect runtime ──────────────────────────────────────────────────
COLAB = "google.colab" in sys.modules
IGWN  = os.path.exists("/cvmfs/oasis.opensciencegrid.org")

REPO_GIT    = "https://github.com/saulo-albuquerque-phys/mlgw_bns_jax.git"
REPO_BRANCH = "blackjax_ns_gw_pe"
REPO_DIR    = "/content/mlgw_bns_jax" if COLAB else os.path.expanduser("~/mlgw_bns_jax")
JENPYROQ_GIT = "git+https://github.com/GCArullo/JenpyROQ.git"

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

def _clone_repo(dest):
    if not os.path.isdir(dest):
        subprocess.check_call([
            "git", "clone", "--branch", REPO_BRANCH, "--depth", "1",
            REPO_GIT, dest,
        ])

def _patch_jenpyroq():
    """Fix JenpyROQ for numpy 2.0 (np.VisibleDeprecationWarning removed)."""
    if not hasattr(np, "VisibleDeprecationWarning"):
        np.VisibleDeprecationWarning = FutureWarning
    try:
        import site
        site_pkgs = site.getsitepackages()[0]
        for fname in ("jenpyroq.py", "__main__.py"):
            fpath = os.path.join(site_pkgs, "JenpyROQ", fname)
            if os.path.isfile(fpath):
                txt = open(fpath).read()
                if "np.VisibleDeprecationWarning" in txt and "getattr" not in txt:
                    txt = txt.replace(
                        "category=np.VisibleDeprecationWarning",
                        'category=getattr(np, "VisibleDeprecationWarning", FutureWarning)',
                    )
                    open(fpath, "w").write(txt)
                    print(f"  Patched {fpath}")
    except Exception as e:
        print(f"  JenpyROQ patch skipped: {e}")

# ── Setup per environment ───────────────────────────────────────────
if COLAB:
    print("=== Google Colab detected ===")
    _pip("h5py", "scikit-learn", "poetry-core", "matplotlib", JENPYROQ_GIT)
    _clone_repo(REPO_DIR)
    os.chdir(REPO_DIR)
    _pip("-e", ".")
    _patch_jenpyroq()

elif IGWN:
    print("=== IGWN JupyterHub detected ===")
    print("Checking if mlgw-bns-jax conda env is active ...")

    if "mlgw-bns-jax" in os.environ.get("CONDA_DEFAULT_ENV", ""):
        print("  ✓ mlgw-bns-jax environment is active.")
    else:
        print("  ⚠ The mlgw-bns-jax kernel is NOT active.")
        print("  → Run 'bash setup_igwn.sh' from a terminal first,")
        print("    then select the 'mlgw-bns-jax' kernel in this notebook.")
        print("")
        print("  Attempting in-kernel setup (may fail with base conflicts) ...")

    # Install JenpyROQ + poetry-core (always safe via pip)
    _pip("poetry-core", JENPYROQ_GIT)
    _pip("setuptools<71")  # ensure pkg_resources exists

    _clone_repo(REPO_DIR)
    os.chdir(REPO_DIR)
    _pip("--no-deps", "-e", ".")
    _patch_jenpyroq()

else:
    print("=== Local machine ===")
    print("Assuming dependencies are already installed.")
    print("If not, run:  pip install h5py scikit-learn poetry-core matplotlib")
    print("              pip install 'JenpyROQ @ git+https://github.com/GCArullo/JenpyROQ.git'")
    print("              pip install -e .")
    _patch_jenpyroq()

# ── Find repo root ─────────────────────────────────────────────────
if not os.path.isfile("config_roq_mlgw_bns_jax_gw170817.ini"):
    for c in [REPO_DIR, os.path.expanduser("~/mlgw_bns_jax"), "/content/mlgw_bns_jax",
              os.path.dirname(os.path.abspath("__file__"))]:
        if os.path.isdir(c) and os.path.isfile(os.path.join(c, "config_roq_mlgw_bns_jax_gw170817.ini")):
            os.chdir(c)
            break

assert os.path.isfile("config_roq_mlgw_bns_jax_gw170817.ini"), \
    "Cannot find config file. Please cd to the mlgw_bns_jax repo root."
assert os.path.isfile("mlgw_bns_roq_wrapper.py"), \
    "Cannot find mlgw_bns_roq_wrapper.py."

print(f"\nWorking directory: {os.getcwd()}")
print(f"Python: {sys.version}")
print(f"NumPy:  {np.__version__}")
print("All required files found. ✓")

## 2. Load JenpyROQ and Read Configuration

We register the NumPy-based  wrapper with JenpyROQ and read the
ROQ configuration file for GW170817.

The configuration specifies:
- Frequency range: 23–2000 Hz
- Segment length: 128 s (→ Δf = 1/128 Hz ≈ 0.0078 Hz → ~253k frequency points)
- Training ranges for chirp mass (1.18–1.21 M☉), mass ratio (1–2),
  spins (±0.5), tidal deformabilities (5–5000)
- Tolerances: 1e-4 (linear), 1e-6 (quadratic)
- 3 enrichment cycles with 10k, 50k, 100k training waveforms

In [ ]:
import os, logging, sys
import numpy as np

# Register the mlgw_bns wrapper (pure NumPy — no JAX needed)
import mlgw_bns_roq_wrapper  # side-effect: registers WfWrapper["mlgw-bns-jax"]

from JenpyROQ.jenpyroq import JenpyROQ
from JenpyROQ.initialise import read_config
from JenpyROQ.parallel import initialize_serial_pool

print("JenpyROQ loaded, NumPy wrapper registered (no JAX).")

# Logging
logger = logging.getLogger("JenpyROQ")
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s  %(message)s"))
    logger.addHandler(handler)

# Read configuration
CONFIG_FILE = "config_roq_mlgw_bns_jax_gw170817.ini"
OUT_DIR     = "./roq_basis_mlgw_bns_jax/"
os.makedirs(OUT_DIR, exist_ok=True)

config_pars, params_ranges, test_values = read_config(CONFIG_FILE, OUT_DIR, logger)

wf_cfg = config_pars["Waveform_and_parametrisation"]
fmin   = wf_cfg["f-min"]
fmax   = wf_cfg["f-max"]
seglen = wf_cfg["seglen"]

print(f"
Frequency range : [{fmin}, {fmax}] Hz")
print(f"Segment length  : {seglen} s  →  Δf = {1/seglen:.4f} Hz")
print(f"Full grid points: {int((fmax - fmin) * seglen) + 1}")
print(f"Training ranges : {params_ranges}")

## 3. Waveform Smoke Test

Generate a single test waveform on the full 253k-point grid to verify
the NumPy wrapper works correctly before starting the multi-hour ROQ build.

In [ ]:
from mlgw_bns_roq_wrapper import WfMLGWBNS

wf = WfMLGWBNS("mlgw-bns-jax")

# GW170817-like parameters
p_test = {
    "m1": 1.365, "m2": 1.365,
    "s1z": 0.0,  "s2z": 0.0,
    "lambda1": 300.0, "lambda2": 300.0,
    "iota": 2.5, "phiref": 0.6,
}

hp_test, hc_test = wf.generate_waveform(p_test, 1.0 / seglen, fmin, fmax, 10.0)
print(f"Waveform OK: hp shape = {hp_test.shape}, max|hp| = {np.max(np.abs(hp_test)):.3e}")
del hp_test, hc_test

## 4. Build the ROQ Basis

This is the main computation. It runs:

1. **Pre-selection** — corner basis + greedy selection of ~100 elements
2. **Enrichment** — 3 cycles with 10k / 50k / 100k random training waveforms

Each cycle generates random waveforms, projects them onto the current basis,
and adds the worst-represented waveform to the basis until the tolerance is met.

**Linear basis** (tolerance 1e-4): used in ⟨d|h⟩ inner products
**Quadratic basis** (tolerance 1e-6): used in ⟨h|h⟩ inner products

> ⏱ This takes **2–6 hours** depending on hardware. The progress is printed
> as each new basis element is added.

In [ ]:
import time, gc

pool = initialize_serial_pool()

sep = "=" * 60
print(sep)
print("Building ROQ basis for mlgw-bns (NumPy — no JAX)")
print(f"  f = [{fmin}, {fmax}] Hz")
print(f"  seglen = {seglen} s")
print(f"  Output: {OUT_DIR}")
print(sep)

t0 = time.time()

with pool as p:
    roq = JenpyROQ(config_pars, params_ranges, distance=10.0, pool=p)

    # ── Phase 1: LINEAR basis ──────────────────────────────────────
    print("
--- Building LINEAR basis ---")
    data_lin = roq.run("lin")
    n_lin = len(data_lin["lin_emp_nodes"])
    print(f"Linear basis: {n_lin} elements")

    # Free linear arrays before quadratic build.
    # All results are already saved to disk by JenpyROQ
    # (roq_basis_mlgw_bns_jax/ROQ_data/linear/*.npy)
    del data_lin
    gc.collect()
    print("(linear data freed from RAM — saved on disk)")

    # ── Phase 2: QUADRATIC basis ───────────────────────────────────
    print("
--- Building QUADRATIC basis ---")
    data_qua = roq.run("qua")
    n_qua = len(data_qua["qua_emp_nodes"])
    print(f"Quadratic basis: {n_qua} elements")

elapsed = time.time() - t0

# Summary
f_full = np.arange(fmin, fmax + 1.0 / seglen, 1.0 / seglen)
n_full = len(f_full)

print(f"
{sep}")
print(f"  Full frequency grid : {n_full} points")
print(f"  Linear basis        : {n_lin} elements  ({n_full / n_lin:.0f}x reduction)")
print(f"  Quadratic basis     : {n_qua} elements  ({n_full / n_qua:.0f}x reduction)")
print(f"  Elapsed time        : {elapsed/3600:.1f} hours")
print(f"  Output directory    : {OUT_DIR}")
print(sep)

## 5. Validation

Check the ROQ representation error on a test waveform.
The linear interpolation error should be below the tolerance (1e-4).

In [ ]:
import matplotlib.pyplot as plt

# ── Reload basis from disk ──────────────────────────────────────────
roq_lin_dir = os.path.join(OUT_DIR, "ROQ_data", "linear")
B_lin     = np.load(os.path.join(roq_lin_dir, "basis_interpolant_linear.npy"))
nodes_lin = np.load(os.path.join(roq_lin_dir, "empirical_nodes_linear.npy"))

roq_qua_dir = os.path.join(OUT_DIR, "ROQ_data", "quadratic")
B_qua     = np.load(os.path.join(roq_qua_dir, "basis_interpolant_quadratic.npy"))
nodes_qua = np.load(os.path.join(roq_qua_dir, "empirical_nodes_quadratic.npy"))

print(f"Linear  ROQ nodes : {len(nodes_lin)} ({n_full / len(nodes_lin):.0f}x speedup)")
print(f"Quadratic ROQ nodes: {len(nodes_qua)} ({n_full / len(nodes_qua):.0f}x speedup)")

# ── Generate test waveform ──────────────────────────────────────────
wf = WfMLGWBNS("mlgw-bns-jax")
p_wf = {
    "m1": 1.365, "m2": 1.365,
    "s1z": 0.0,  "s2z": 0.0,
    "lambda1": 300.0, "lambda2": 300.0,
    "iota": 2.5, "phiref": 0.6,
}
deltaF = 1.0 / seglen
hp_full, _ = wf.generate_waveform(p_wf, deltaF, fmin, fmax, 10.0)

# ── Normalise and compare ──────────────────────────────────────────
from JenpyROQ.linear_algebra import normalise_vector, scalar_product

hp_norm    = normalise_vector(hp_full, deltaF)
hp_roq_lin = np.dot(B_lin, hp_norm[nodes_lin])
residual   = hp_norm - hp_roq_lin
eie        = scalar_product(residual, residual, deltaF)
tol_lin    = config_pars["ROQ"]["tolerance-lin"]

print(f"
Linear interpolation error: {float(np.real(eie)):.2e}  (tolerance: {tol_lin})")

freq = np.arange(fmin, fmax + deltaF, deltaF)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={"height_ratios": [3, 1]})

ax = axes[0]
ax.plot(freq, np.real(hp_norm), lw=0.8, alpha=0.7, label="Full waveform")
ax.plot(freq, np.real(hp_roq_lin), lw=0.5, ls="--", label="ROQ reconstruction")
ax.scatter(freq[nodes_lin], np.real(hp_norm)[nodes_lin],
           s=8, c="red", zorder=5, label=f"Empirical nodes ({len(nodes_lin)})")
ax.set_ylabel(r"$\Re[	ilde{h}_+]$ (normalised)")
ax.set_title("Linear ROQ vs Full waveform")
ax.legend()

ax = axes[1]
ax.plot(freq, np.real(residual), lw=0.5, color="darkred")
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel("Residual")
ax.set_title(f"Representation error = {float(np.real(eie)):.2e}")

fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "roq_validation_linear.png"), dpi=150)
plt.show()
print("Validation plot saved.")

## 6. Output Summary

List all ROQ data files and their sizes.

In [ ]:
print("ROQ output files:")
print("=" * 60)
for root, dirs, files in os.walk(os.path.join(OUT_DIR, "ROQ_data")):
    level = root.replace(OUT_DIR, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (level + 1)
    for file in sorted(files):
        fpath = os.path.join(root, file)
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"{subindent}{file}  ({size_mb:.1f} MB)")

## 7. Download the ROQ Basis

Package the ROQ output for download (Colab) or note the path (IGWN/local).

After downloading, you can use the ROQ basis for parameter estimation
by running the  notebook.

In [ ]:
import shutil

zip_name = "roq_basis_mlgw_bns_jax"
shutil.make_archive(zip_name, "zip", ".", "roq_basis_mlgw_bns_jax")
print(f"Created {zip_name}.zip ({os.path.getsize(zip_name + '.zip') / 1e6:.1f} MB)")

if COLAB:
    from google.colab import files
    files.download(f"{zip_name}.zip")
    print("Download started.")
else:
    print(f"Zip ready at: {os.path.abspath(zip_name + '.zip')}")
    print(f"
ROQ data directory: {os.path.abspath(OUT_DIR)}")
    print("
Next step: run roq_pe_mlgw_bns_jax.ipynb for parameter estimation.")